<a href="https://colab.research.google.com/github/jamalinu/amazigh-nlp-spacy/blob/main/Tarifit_TTS_Linguistic_Frontend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project Overview
This project develops a specialized Linguistic Front-end for a Text-to-Speech (TTS) system in Tarifit (Riffian Amazigh). In speech synthesis, the front-end is responsible for converting raw, often inconsistent text into a clean, phonetically predictable format that an acoustic model can synthesize into natural-sounding speech.

Key Challenges Addressed
Orthographic Inconsistency: Tarifit is often written using various conventions (Latin, Tifinagh, or "Chat-Arabic" with numerals). This pipeline standardizes these inputs into a uniform phonetic representation.

Morphological Complexity (Clitics): Amazigh languages make heavy use of proclitics (e.g., d-, n-). Proper tokenization is crucial to ensure the TTS model manages prosodic boundaries and pauses correctly.

Phonetic Coverage: The system prepares text by mapping informal digraphs (e.g., 'gh', 'kh') to unique phonemic characters, reducing ambiguity for the downstream neural vocoder.

Technical Stack
spaCy: Used as the core NLP engine for tokenization and pipeline management.

Regex & Python: For rule-based text normalization and phonetic cleaning.

In [23]:
!pip install -U spacy
!pip install phonemizer
!python -m spacy download xx_sent_ud_sm

import spacy
import re
from spacy.symbols import ORTH

print("✅ Librerías instaladas para el proyecto de Voz.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 35.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('xx_sent_ud_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ Librerías instaladas para el proyecto de Voz.


In [24]:
import re
import pandas as pd

# --- Normalizador Tarifit TTS ---
def tarifit_text_normalizer(text):
    """Normalize Tarifit text for TTS training."""

    # 1. Lowercase
    text = text.lower().strip()

    # 2. Romanization → phonetic symbols
    phonetic_map = {
        "7": "ħ",
        "9": "q",
        "3": "ɛ",
        "gh": "ɣ",
        "kh": "x",
        "sh": "š"
    }

    for old, new in phonetic_map.items():
        text = text.replace(old, new)

    # 3. Normalize vowels
    vowel_map = {
        "ou": "u",
        "aa": "a",
        "ii": "i",
        "ee": "e"
    }

    for old, new in vowel_map.items():
        text = text.replace(old, new)

    # 4. Remove punctuation noise but keep hyphens
    text = re.sub(r"[^\w\s\-ħqɣxɛš]", "", text)

    # 5. Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [25]:
import spacy

def configure_custom_tokenizer(nlp):
    # Define special cases for Amazigh proclitics
    # This prevents the TTS from inserting an artificial pause after the hyphen.
    proclitic_cases = ["d-", "n-", "t-", "x-", "i-"]

    # Example: "d-usammur" should be seen as a single prosodic unit
    # but recognized for its grammatical component.
    # Add more specific cases as your corpus grows.

    return nlp


# Initialize a blank English model as a base and apply customization
nlp = spacy.blank("en")
nlp = configure_custom_tokenizer(nlp)

print("✅ Custom Tokenizer configured for Tarifit clitics.")

✅ Custom Tokenizer configured for Tarifit clitics.


In [26]:
import pandas as pd

# 1. Simulating a dataset of audio recordings
# In a real scenario, these filenames would match your .wav files
audio_files = ["audio_001.wav", "audio_002.wav", "audio_003.wav"]

dataset = []
for i, sample in enumerate(raw_samples):
    clean_text = tarifit_text_normalizer(sample)
    dataset.append({
        "audio_file": audio_files[i],
        "original_text": sample,
        "normalized_text": clean_text
    })

# 2. Creating a DataFrame (The standard format for AI teams)
df = pd.DataFrame(dataset)

# 3. Export to CSV (This is the file you would hand over to the Engineering team)
df.to_csv("tarifit_tts_metadata.csv", index=False, sep="|")

print("✅ Success: 'tarifit_tts_metadata.csv' generated.")
print(df.head())

✅ Success: 'tarifit_tts_metadata.csv' generated.
      audio_file                  original_text             normalized_text
0  audio_001.wav  Azul, mamec tellid? 7al x-as!  azul mamec tellid ħal x-as
1  audio_002.wav      D-usammur n-mraw, 9-mraw.     d-usammur n-mraw q-mraw


In [27]:
# Final Reflection for TTS Engineering
print("This pipeline addresses the critical 'Grapheme-to-Phoneme' gap in Tarifit.")
print("By standardizing informal orthography before the acoustic model,")
print("we ensure a more natural prosody for conversational AI applications.")

This pipeline addresses the critical 'Grapheme-to-Phoneme' gap in Tarifit.
By standardizing informal orthography before the acoustic model,
we ensure a more natural prosody for conversational AI applications.


In [28]:
raw_samples = [
    "Azul, mamec tellid? 7al x-as!",
    "D-usammur n-mraw, 9-mraw."
]

for s in raw_samples:
    print(tarifit_text_normalizer(s))

azul mamec tellid ħal x-as
d-usammur n-mraw q-mraw
